# S7 multi-doc 묶음 구성

multi-doc 질의(SPEC "보조 관찰: multi-doc 질의")의 후보 집합과 시작 묶음을 확인하고, 유형·크기 상한·표본 수·시작 묶음 크기를 정하는 근거 자료다(`docs/slices/07-bundles.md`).

- 묶음은 라벨 메타데이터(회의, `law`, `questioner_ID`+위원회)로만 만든다. 검색기 점수와 LLM을 쓰지 않는다
- `test` 분할 회의의 문서는 뺀다. 질의(`summary_q`)는 읽지 않는다
- 산출물은 `rag.multidoc.pools`가 만든 `data/multidoc/pools.jsonl`이고, 이 노트북은 같은 함수로 다시 계산해 확인한다

In [1]:
import json
from pathlib import Path

import pandas as pd

from rag.index import load_cfg
from rag.ingest import iter_labels
from rag.multidoc.pools import build, collect, find_pools

In [2]:
ROOT = Path.cwd().parent
cfg = load_cfg()
M = cfg.multidoc
TYPES = list(M.types)
POOLS = ROOT / cfg.paths.multidoc_pools
CAPS = [8, 12, 16, 20, 30, 50]  # 비교할 후보 집합 크기 상한
N_EXAMPLES = 2  # 유형별로 출력할 후보 집합 예시 수
N_DOCS = 3  # conf·questioner 예시에서 출력할 문서 수
dict(M)

{'types': ['conf', 'law', 'questioner'],
 'pool_max_docs': 20,
 'n_pools_per_type': 250,
 'seed_sizes': [2, 3]}

## 1. 메타데이터

In [3]:
splits = pd.read_csv(ROOT / cfg.paths.splits, dtype=str)
test_confs = set(splits.loc[splits["split"] == "test", "conference_number"])
rows = collect((r for _, r in iter_labels(ROOT / cfg.paths.raw_dir)), test_confs)
print(rows.shape, "문서", rows["doc_id"].nunique(), "회의", rows["conf"].nunique(),
      "| 뺀 test 회의", len(test_confs))
rows.head(3)

(36630, 5) 문서 35615 회의 8908 | 뺀 test 회의 754


,doc_id,conf,law,questioner,committee
0,029172-f2ecda8948,029172,,547,통일외교통상위원회
1,030045-7952325e15,030045,,3337,건설교통위원회
2,029414-42012a05b5,029414,,1798,보건복지위원회


In [4]:
empty = {c: (rows[c] == "").mean() for c in ["law", "questioner", "committee"]}
pd.Series(empty, name="빈 값 비율").round(3)

law           0.594
questioner    0.007
committee     0.000
Name: 빈 값 비율, dtype: float64

## 2. 유형별 후보 집합 크기 (상한 없음)

유형 조건(문서 2개 이상, `law`·`questioner`는 회의 2개 이상)만 적용한 분포다.

In [5]:
all_pools = {k: find_pools(rows, k, 10**6) for k in TYPES}
sizes = {k: pd.Series([len(d) for _, d in p]) for k, p in all_pools.items()}
pd.DataFrame({k: s.describe(percentiles=[0.5, 0.75, 0.9, 0.99]) for k, s in sizes.items()}).round(1)

,conf,law,questioner
count,2184.0,437.0,4330.0
mean,13.2,13.7,7.2
std,6.4,81.7,7.7
min,2.0,2.0,2.0
50%,14.0,4.0,5.0
75%,17.0,9.0,9.0
90%,18.0,24.4,15.0
99%,38.2,99.7,39.7
max,60.0,1679.0,101.0


## 3. 크기 상한별 남는 후보 집합 수

완전성 검사는 후보 집합 전체를 읽으므로 상한이 검사 입력 크기를 정한다. 괄호 안은 전체 대비 비율이다.

In [6]:
pd.DataFrame({k: {cap: f"{(s <= cap).sum()} ({(s <= cap).mean():.0%})" for cap in CAPS}
              for k, s in sizes.items()}).rename_axis("상한")

,conf,law,questioner
상한,,,
8,493 (23%),320 (73%),3202 (74%)
12,867 (40%),359 (82%),3718 (86%)
16,1595 (73%),374 (86%),3974 (92%)
20,2066 (95%),385 (88%),4095 (95%)
30,2154 (99%),405 (93%),4235 (98%)
50,2182 (100%),422 (97%),4315 (100%)


## 4. 생성 결과 확인

`pools.jsonl`이 같은 함수·설정으로 다시 계산한 결과와 같은지, 지시서 규칙을 지키는지 본다.

In [7]:
pools = [json.loads(line) for line in POOLS.open(encoding="utf-8")]
rebuilt, available = build(rows, TYPES, M.pool_max_docs, M.n_pools_per_type,
                           list(M.seed_sizes), cfg.seed)
print("파일과 재계산 일치:", pools == rebuilt, "| 가용 후보 집합 수", available)
df = pd.DataFrame(pools)
df["size"] = df["doc_ids"].str.len()
df["seed_size"] = df["seed_doc_ids"].str.len()
df.groupby("type", sort=False).agg(
    후보집합=("pool_id", "size"), 크기_중앙값=("size", "median"),
    크기_최대=("size", "max"), 시작묶음_평균=("seed_size", "mean")).round(2)

파일과 재계산 일치: True | 가용 후보 집합 수 {'conf': 2066, 'law': 385, 'questioner': 4095}


,후보집합,크기_중앙값,크기_최대,시작묶음_평균
type,,,,
conf,250,14.0,20,0.00
law,250,3.0,20,2.22
questioner,250,4.0,19,0.00


In [8]:
conf_of = rows.drop_duplicates("doc_id").set_index("doc_id")["conf"]
all_docs = df.explode("doc_ids")["doc_ids"]
seeds = list(zip(df["type"], df["seed_doc_ids"], df["doc_ids"], strict=True))
law = [(s, d) for t, s, d in seeds if t == "law"]
{
    "test 회의 문서": int(conf_of[all_docs].isin(test_confs).sum()),
    "law 시작 묶음이 후보 집합 밖": sum(not set(s) <= set(d) for s, d in law),
    "law 시작 묶음의 회의 중복": sum(conf_of[s].nunique() < len(s) for s, _ in law),
    "conf·questioner 시작 묶음이 있음": sum(len(s) > 0 for t, s, _ in seeds if t != "law"),
}

{'test 회의 문서': 0,
 'law 시작 묶음이 후보 집합 밖': 0,
 'law 시작 묶음의 회의 중복': 0,
 'conf·questioner 시작 묶음이 있음': 0}

In [9]:
types_of = df.explode("doc_ids").groupby("doc_ids")["type"].nunique()
print(f"선택된 후보 집합의 문서 {len(types_of)}개, 두 유형 이상에 든 문서 {(types_of > 1).sum()}개")

선택된 후보 집합의 문서 5592개, 두 유형 이상에 든 문서 245개


## 5. 시작 묶음과 후보 집합 예시

`law`는 S7이 고른 시작 묶음을 보인다. `conf`·`questioner`는 S7이 시작 묶음을 고르지 않고 S8 생성 에이전트가 후보 집합에서 고르므로, 후보 집합의 문서 앞 `N_DOCS`개를 보인다. 회의록 발언 원문은 공개 자료라 앞부분을 출력한다.

In [10]:
examples = df.groupby("type", sort=False).head(N_EXAMPLES).copy()
cols = zip(examples["type"], examples["seed_doc_ids"], examples["doc_ids"], strict=True)
examples["shown"] = [s if t == "law" else d[:N_DOCS] for t, s, d in cols]
need = set(examples["shown"].sum())
corpus = {}
with (ROOT / cfg.paths.corpus).open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        if d["doc_id"] in need:
            corpus[d["doc_id"]] = d
len(corpus)

15

In [11]:
for _, p in examples.iterrows():
    shown = "시작 묶음" if p["type"] == "law" else f"후보 집합 앞 {len(p['shown'])}개"
    print(f"=== {p['pool_id']} | 키 {p['key']} | 후보 집합 {p['size']}개 | {shown}")
    for doc_id in p["shown"]:
        d = corpus[doc_id]
        print(f"- {doc_id} | {d['date']} | {d['committee_name']} | 안건 {d['agenda'][:60]}")
        print("  ", " ".join(d["context"][:200].split()))
    print()

=== conf-0000 | 키 048062 | 후보 집합 10개 | 후보 집합 앞 3개
- 048062-0a6464ac01 | 2018년3월6일(화) | 국토교통위원회-국토법안심사소위원회 | 안건 5. 도시 및 주거환경정비법 일부개정법률안(신상진 의원 대표발의)(계속)
   이게 민홍철 위원님이 말씀하셨듯이 성남 대이주단지 이런 것들을 대상으로 한 법인가요? 주택정비과장입니다. 잠깐 말씀드리면 그 개념에 대해서는 현재 시행령에서 정하고 있습니다. 다른 공익사업을 통해서 300세대 이상이 특정 지역으로 집단이주가 발생된 그런 지역들을 얘기하고, 신상진 의원님이 얘기하시는 성남에 있는 2개 지역 거기는 모두 시행령 개정에 포함됩
- 048062-149a721167 | 2018년3월6일(화) | 국토교통위원회-국토법안심사소위원회 | 안건 6. 공익사업을 위한 토지 등의 취득 및 보상에 관한 법률 일부개정법률안(이춘석 의원 대표발의)
   이게 지금 여기에서만 문제되는 것이 아니고 유사한 일에서 다 문제가 될 텐데, 빈집 및 소규모주택 정비에 관한 특례법에 의하면 어떤 요건들이 갖추어지면 할 것 아닙니까, 그렇지요? 빈집 및 소규모주택정비사업에 의해서는 토지수용을 할 수가 없습니다. 없는데, 수용 근거를 마련해 주려고 하는 법안인데…… 그렇지 않습니다. 수용 근거는 빈집법에 수용 근거가 있어
- 048062-1c7457b0b7 | 2018년3월6일(화) | 국토교통위원회-국토법안심사소위원회 | 안건 8. 부동산투자회사법 일부개정법률안(주승용 의원 대표발의)9. 부동산개발업의 관리 및 육성에 관한 법률 일부
   위원님들의 만장일치…… 혹시 이 문제에 대해서 주승용 의원님실과 특별히 상의해 보신 적이 있으십니까? 특별히 이 건에 대해서 상의하지는 않았습니다. 아주 큰 관심 법안은 아니시지요? 예.

=== conf-0001 | 키 050499 | 후보 집합 3개 | 후보 집합 앞 3개
- 050499-6d1d908826 | 2020년11월19일(목) | 보건복지위원회

## 6. 후보 집합 안의 주제 연결

5절에서 `law` 시작 묶음은 같은 법안을 다른 시기에 다뤄 주제가 이어진다. `conf`·`questioner` 후보 집합은 그렇지 않을 수 있다. conf-0000은 한 소위원회 회의에서 차례로 심사한 서로 다른 법안(안건 5·6·8)이고, questioner-0000은 같은 의원이 한 예산 심사에서 다룬 서로 다른 항목(해양 쓰레기 조사, 귀어학교)이다. 발언 순서상 연속 구간이나 무작위로 고르면 주제가 이어지지 않는 시작 묶음이 나올 수 있다.

같은 법안으로 범위를 좁히면 주제 연결을 보장할 수 있다. 그런 묶음이 얼마나 있는지 센다.


In [12]:
with_law = rows[rows["law"] != ""]
conf_law = with_law.groupby(["conf", "law"])["doc_id"].nunique()
print("같은 회의에서 같은 법안을 다룬 문서 2개 이상인 (회의, 법안):", (conf_law >= 2).sum(),
      "| 회의 수", conf_law[conf_law >= 2].index.get_level_values(0).nunique())
q_law = with_law[with_law["questioner"] != ""].groupby(["questioner", "law"]).agg(
    n=("doc_id", "nunique"), c=("conf", "nunique"))
print("질의자+법안, 회의 2개 이상·크기 20 이하:", ((q_law["c"] >= 2) & (q_law["n"] <= 20)).sum())

같은 회의에서 같은 법안을 다룬 문서 2개 이상인 (회의, 법안): 1422 | 회의 수 1084
질의자+법안, 회의 2개 이상·크기 20 이하: 667


법안으로 좁히면 법안 값이 있는 문서만 남는다. 회의구분별로 얼마나 남는지 본다.


In [13]:
meeting = {}
with (ROOT / cfg.paths.corpus).open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        meeting[d["doc_id"]] = d["meeting_name"]
docs = rows.drop_duplicates("doc_id").copy()
docs["meeting"] = docs["doc_id"].map(meeting)
law_docs = set(with_law["doc_id"])
has_law = docs["doc_id"].isin(law_docs)
pd.concat({"전체 문서": docs["meeting"].value_counts(normalize=True),
           "법안 값 있는 문서": docs.loc[has_law, "meeting"].value_counts(normalize=True),
           "법안 값 있는 비율": has_law.groupby(docs["meeting"]).mean()},
          axis=1).round(3)

,전체 문서,법안 값 있는 문서,법안 값 있는 비율
meeting,,,
국정감사,0.412,0.122,0.116
소위원회,0.404,0.784,0.762
특별위원회,0.100,0.054,0.213
예산결산특별위원회,0.053,0.021,0.154
본회의,0.030,0.020,0.254


## 7. 정리

- 같은 법안으로 좁히면 주제 연결은 보장되지만, 법안 값이 있는 문서는 소위원회가 78%(전체 40%), 국정감사가 12%(전체 41%)라 세트가 법안 심사 회의로 쏠린다
- 그래서 `law`만 S7이 시작 묶음을 고르고, `conf`·`questioner`는 S8 생성 에이전트가 후보 집합의 개요를 보고 서로 이어지는 문서를 고르며 없으면 후보 집합을 버리도록 정했다(사용자 결정, DECISIONS D-13). LLM이 고르는 것은 SPEC이 금지한 검색기 점수 선택에 해당하지 않는다
